
# Dia 3 — Web Scraping: Tabelas HTML + Prática
**Data:** 29/05/2026 | **Horário:** 19h – 22h

---

## O que vamos aprender hoje?

1. Revisão rápida do Dia 2
2. `pandas.read_html()` — o atalho para tabelas
3. Scraping de tabelas da Wikipedia
4. `quotes.toscrape.com` — prática com citações
5. Visualizando e salvando os dados coletados



---
## 🕐 PARTE 1 — Revisão Rápida (19h – 19h30)

### O que aprendemos até agora?

| Dia | Conteúdo | Ferramentas |
|-----|----------|------------|
| Dia 1 | Dados públicos (.gov) e Kaggle | pandas, matplotlib, requests |
| Dia 2 | Web Scraping básico | requests, BeautifulSoup |
| Dia 3 | Tabelas HTML + Prática | pandas.read_html, tudo junto |

### Revisão: BeautifulSoup em 3 passos
```python
# Passo 1: Buscar a página
resposta = requests.get(url)

# Passo 2: Analisar o HTML
soup = BeautifulSoup(resposta.text, 'html.parser')

# Passo 3: Extrair dados
elementos = soup.find_all('tag', class_='classe')
```


In [ ]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import matplotlib.pyplot as plt
import time

print('Bibliotecas importadas!')

---
## 🕐 PARTE 2 — pandas.read_html(): o atalho mágico (19h30 – 20h)

### O pandas tem uma função INCRÍVEL para tabelas HTML!

Quando um site tem uma `<table>` HTML, o pandas consegue ler diretamente — **sem precisar de BeautifulSoup!**

```python
# Lê TODAS as tabelas de uma URL
tabelas = pd.read_html(url)

# tabelas é uma lista — cada item é uma tabela diferente
primeira_tabela = tabelas[0]
```

### Quando usar `read_html` vs BeautifulSoup?

| Situação | Use |
|----------|-----|
| Dados estão em uma `<table>` HTML | `pd.read_html()` — mais simples! |
| Dados estão em `<div>`, `<span>`, etc. | `BeautifulSoup` |
| Página JavaScript dinâmica | Selenium (avançado) |

In [ ]:

# Exemplo: ler tabela da Wikipedia sobre os países mais populosos

url_wiki = 'https://pt.wikipedia.org/wiki/Lista_de_pa%C3%ADses_por_popula%C3%A7%C3%A3o'

# Precisamos enviar um cabeçalho para a Wikipedia não bloquear a requisição
headers = {'User-Agent': 'Mozilla/5.0'}
resposta_wiki = requests.get(url_wiki, headers=headers)

# read_html lê direto do HTML que já buscamos
tabelas = pd.read_html(resposta_wiki.text)

print(f'Tabelas encontradas na página: {len(tabelas)}')


In [ ]:

# Pegar a primeira tabela
df_paises = tabelas[0]

print('Colunas da tabela:')
print(df_paises.columns.tolist())
print()
df_paises.head(10)


In [ ]:
# Verificar o formato dos dados
df_paises.info()

In [ ]:

# Pegar as 3 primeiras colunas (posição, país, população)
df_pop = df_paises.iloc[:, :3].copy()
df_pop.columns = ['Posicao', 'Pais', 'Populacao']

# Remover linhas vazias e pegar os 15 primeiros
df_pop = df_pop.dropna()
df_pop = df_pop.head(15)

print('Dados prontos!')
df_pop


---
## 🕑 PARTE 3 — Scraping da Wikipedia com BeautifulSoup (20h – 20h45)

Vamos coletar dados sobre os **maiores países do mundo por área** diretamente da Wikipedia.

In [ ]:
# Scraping da Wikipedia: lista de países por área
url = 'https://pt.wikipedia.org/wiki/Lista_de_pa%C3%ADses_por_%C3%A1rea'

headers = {
    'User-Agent': 'Mozilla/5.0 (Educational scraping project)'
}

resposta = requests.get(url, headers=headers)
print(f'Status: {resposta.status_code}')

In [ ]:

# Ler as tabelas direto do HTML que já buscamos
tabelas_area = pd.read_html(resposta.text)
print(f'Tabelas encontradas: {len(tabelas_area)}')

# Mostrar a primeira tabela
print()
print('Primeira tabela:')
tabelas_area[0].head(5)


---
## 🕑 PARTE 4 — Quotes to Scrape: Prática com BeautifulSoup (20h45 – 21h15)

Vamos coletar **citações famosas** do site `quotes.toscrape.com`

In [ ]:
# Coletar citações de quotes.toscrape.com
url_quotes = 'http://quotes.toscrape.com/'

resposta = requests.get(url_quotes)
soup = BeautifulSoup(resposta.text, 'html.parser')

# Cada citação está em um <div class='quote'>
citacoes_html = soup.find_all('div', class_='quote')
print(f'Citações encontradas: {len(citacoes_html)}')

In [ ]:

# Ver a estrutura HTML de uma citação (limitado para não poluir a tela)
print('Estrutura HTML da primeira citação:')
print(citacoes_html[0].prettify()[:400])


In [ ]:

# Extrair todas as citações da primeira página
lista_citacoes = []

for citacao in citacoes_html:
    # Texto da citação
    texto = citacao.find('span', class_='text').text

    # Autor
    autor = citacao.find('small', class_='author').text

    # Tags — coletar cada uma em um loop
    lista_tags = []
    for tag in citacao.find_all('a', class_='tag'):
        lista_tags.append(tag.text)
    tags_str = ', '.join(lista_tags)

    lista_citacoes.append({
        'texto': texto,
        'autor': autor,
        'tags':  tags_str
    })

df_quotes = pd.DataFrame(lista_citacoes)
print(f'Citações coletadas: {len(df_quotes)}')
df_quotes


In [ ]:

# Coletar 3 páginas de citações
todas_citacoes = []

for pagina in range(1, 4):
    url_pag = f'http://quotes.toscrape.com/page/{pagina}/'
    resp    = requests.get(url_pag)
    soup_pag = BeautifulSoup(resp.text, 'html.parser')
    citacoes_pag = soup_pag.find_all('div', class_='quote')

    for c in citacoes_pag:
        texto = c.find('span', class_='text').text
        autor = c.find('small', class_='author').text

        lista_tags = []
        for tag in c.find_all('a', class_='tag'):
            lista_tags.append(tag.text)
        tags_str = ', '.join(lista_tags)

        todas_citacoes.append({'texto': texto, 'autor': autor, 'tags': tags_str})

    print(f'Página {pagina}: {len(citacoes_pag)} citações')
    time.sleep(0.5)

df_citacoes = pd.DataFrame(todas_citacoes)
print(f'\nTotal: {len(df_citacoes)} citações coletadas')


In [ ]:
# Autores com mais citações
print('Top 5 autores com mais citações:')
print(df_citacoes['autor'].value_counts().head(5))


---
## 🕒 PARTE 5 — Prática: Visualizando os Dados Coletados (21h15 – 22h)

Já temos dois DataFrames prontos com dados reais que coletamos nessa aula:
- `df_citacoes` — citações de `quotes.toscrape.com`
- `tabelas_area` — países por área da Wikipedia

Vamos explorar e visualizar esses dados!


In [ ]:

# Quantas citações temos por autor?
contagem_autores = df_citacoes['autor'].value_counts()

print('Top 10 autores com mais citações:')
print(contagem_autores.head(10))


In [ ]:

# Gráfico: top 5 autores com mais citações
top5 = df_citacoes['autor'].value_counts().head(5)

plt.figure(figsize=(9, 5))
plt.bar(top5.index, top5.values, color='steelblue')
plt.title('Top 5 Autores — quotes.toscrape.com')
plt.xlabel('Autor')
plt.ylabel('Numero de citacoes')
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()


In [ ]:

# Salvar as citações coletadas em CSV
df_citacoes.to_csv('./dados/citacoes.csv', index=False)
print('Citacoes salvas em: ./dados/citacoes.csv')
print(f'Total de registros: {len(df_citacoes)}')



---
## ✏️ Exercícios

**Exercício 1:** Mostre todos os **autores únicos** presentes em `df_citacoes` (use `.unique()`).

**Exercício 2:** Filtre o `df_citacoes` e mostre apenas as citações de **Albert Einstein**.

**Exercício 3:** Com os dados do Wikipedia (`tabelas_area[0]`), mostre as 10 primeiras linhas da tabela de países por área.

**Exercício 4 (desafio):** Crie um gráfico de pizza mostrando a proporção de citações por autor (use os top 5).


In [ ]:

# Exercício 1 — Resolução
autores_unicos = df_citacoes['autor'].unique()
print(f'Total de autores diferentes: {len(autores_unicos)}')
print(autores_unicos)



---
## 📋 Resumo Final do Curso

### Em 3 aulas, você aprendeu:

| Dia | Conteúdo | Ferramentas |
|-----|----------|-------------|
| 26/05 | Dados .gov e Kaggle | pandas, matplotlib, requests |
| 28/05 | Web Scraping básico | requests, BeautifulSoup |
| 29/05 | Tabelas HTML + Prática | pd.read_html, tudo integrado |

### Próximos passos sugeridos:
- Explorar o **Kaggle** e participar de competições para iniciantes
- Aprender **Selenium** para scraping de páginas dinâmicas (JavaScript)
- Aprender sobre **APIs** — outra forma de coletar dados estruturados
- Estudar **SQL** para trabalhar com bancos de dados

### Recursos para continuar:
- 📚 docs.python-requests.org — documentação do requests
- 📚 beautiful-soup-4.readthedocs.io — documentação do BeautifulSoup
- 📚 pandas.pydata.org — documentação do pandas
- 🎮 toscrape.com — mais exercícios de scraping
- 🏆 kaggle.com — datasets e competições



---
## 📋 Resumo Final do Curso

### Em 3 aulas, você aprendeu:

| Dia | Conteúdo | Ferramentas |
|-----|----------|-------------|
| 26/05 | Dados .gov e Kaggle | pandas, matplotlib, requests |
| 28/05 | Web Scraping básico | requests, BeautifulSoup |
| 29/05 | Tabelas + Projeto Final | pd.read_html, tudo integrado |

### Próximos passos sugeridos:
- Explorar o **Kaggle** e participar de competições para iniciantes
- Aprender **Selenium** para scraping de páginas dinâmicas (JavaScript)
- Aprender sobre **APIs** — outra forma de coletar dados estruturados
- Estudar **SQL** para trabalhar com bancos de dados

### Recursos para continuar:
- 📚 docs.python-requests.org — documentação do requests
- 📚 beautiful-soup-4.readthedocs.io — documentação do BeautifulSoup
- 📚 pandas.pydata.org — documentação do pandas
- 🎮 toscrape.com — mais exercícios de scraping
- 🏆 kaggle.com — datasets e competições
